## DGM American Option Pricing
This notebook applies the DGM method to solve the Black-Scholes PDE for an American call option on the best performing asset from a basket of 5 assets. 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import numpy as np
from torch.distributions.normal import Normal
import copy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
if torch.cuda.is_available():
    torch.cuda.init()
    _ = torch.empty(1, device="cuda")

### Hyperparameters and Constants

In [ ]:
# Option Parameters
r = 0.05           # risk-free rate
K = 100.0          # strike
T = 1.0            # maturity
S_max = 200.0      # maximum price used in domain
S_min = 0.0
no_assets = 3
sigmas = [0.2, 0.2, 0.2]

In [ ]:
# Neural network training parameters
no_hidden = 4       # number of hidden layers
no_neurons = 64     # number of neurons per hidden layer
lr = 1e-3          # learning rate
no_epochs = 2000   # number of training epochs
no_loops = 10 # number of iterations before resampling
bd_batch_size = 2**8  # batch size for sampling
int_batch_size = 2**10  # batch size for sampling

In [ ]:
# Loss weights
lambda_pde = 1.0
lambda_term = 1.0
lambda_am = 1.0

### Neural Network Model

In [ ]:
class DGMNet(nn.Module):
    def __init__(self, n_assets, n_hidden, n_neurons):
        super(DGMNet, self).__init__()
        self.n_assets = n_assets
        
        in_dim = 1 + n_assets
        out_dim = 1
        
        layers = []
        layers.append(nn.Linear(in_dim, n_neurons))
        layers.append(nn.Tanh())
        for _ in range(n_hidden - 1):
            layers.append(nn.Linear(n_neurons, n_neurons))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(n_neurons, out_dim))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, t, S):
        x = torch.cat([t, S], dim=1)
        return self.model(x)

### Boundary Conditions

In [ ]:
def boundary_lower(t):
    return 0.0

In [ ]:
def terminal(S):
    S_best = torch.max(S, dim=1, keepdim=True)[0]
    return torch.clamp(S_best - K, min=0.0)

### Loss Function

In [ ]:
def pde_residual(model, t, S, sigmas, r):
    t.requires_grad_(True)
    S.requires_grad_(True)
    
    V = model(t, S)
    
    # dV/dt
    dV_dt = torch.autograd.grad(
        V, t, 
        grad_outputs=torch.ones_like(V), 
        retain_graph=True, create_graph=True
    )[0]
    
    pde = dV_dt
    
    batch_size, n_assets = S.shape
    for i in range(n_assets):
        sigma_i = sigmas[i]
        # dV/dS_i
        dV_dSi = torch.autograd.grad(
            V, S, 
            grad_outputs=torch.ones_like(V),
            retain_graph=True, create_graph=True
        )[0][:, i:i+1]
        
        # d2V/dS_i^2
        d2V_dSi2 = torch.autograd.grad(
            dV_dSi, S,
            grad_outputs=torch.ones_like(dV_dSi),
            retain_graph=True, create_graph=True
        )[0][:, i:i+1]
        
        Si = S[:, i:i+1]
        pde += r * Si * dV_dSi + 0.5 * sigma_i**2 * Si**2 * d2V_dSi2
    
    pde = pde - r * V
    return pde

In [ ]:
def american_constraint_loss(model, t, S):
    S_best = torch.max(S, dim=1, keepdim=True)[0]
    payoff =  torch.clamp(S_best - K, min=0.0)
    V = model(t, S)
    return torch.mean((torch.max(payoff, V) - V)**2)

In [ ]:
def terminal_loss(net, s_terminal):
    tT = torch.ones((s_terminal.shape[0], 1), device=device) * T
    V_T = net(tT, s_terminal)
    payoff_T = terminal(s_terminal)
    return torch.mean((V_T - payoff_T)**2)

### Training Loop

In [ ]:
def train_model(model, optimizer, no_epochs, int_batch_size, 
    bd_batch_size, device=device, patience=100, alpha=0.5
):
    # Record of various losses
    losses = {'res': [], 'am': [], 'term': [], 'total': []}

    # Track the best (lowest) total loss and corresponding model weights
    lowest_loss = float('inf')
    best_model_weights = copy.deepcopy(model.state_dict())
    
    # Track epochs since last improvement
    epochs_since_improvement = 0

    pbar = tqdm(range(no_epochs))
    for epoch in pbar:
        current_model = copy.deepcopy(model)
        
        # Sample interior points for PDE residual
        t_int = torch.rand(int_batch_size, 1, device=device) * T
        s_int = torch.rand(int_batch_size, no_assets, device=device) * (S_max - S_min) + S_min
        s_terminal = torch.rand(bd_batch_size, no_assets, device=device) * (S_max - S_min) + S_min

        for i in range(no_loops):
            optimizer.zero_grad()
        
            # PDE residual
            pde_val = pde_residual(model, t_int, s_int, sigmas=sigmas, r=r)
            loss_pde = torch.mean(pde_val**2)
            
            # American loss
            loss_am = american_constraint_loss(current_model, t_int, s_int)
            
            # Terminal condition
            loss_term = terminal_loss(model, s_terminal)
        
            # Combine losses
            loss = (lambda_pde * loss_pde 
                    + lambda_am * loss_am
                    + lambda_term * loss_term)
            
            # Backpropagation
            loss.backward()
            optimizer.step()
        
            # Store losses
            losses['res'].append(loss_pde.item())
            losses['am'].append(loss_am.item())
            losses['term'].append(loss_term.item())
            losses['total'].append(loss.item())
        
        # Check if this is the best model so far
        if loss.item() < lowest_loss:
            lowest_loss = loss.item()
            best_model_weights = copy.deepcopy(model.state_dict())
            epochs_since_improvement = 0  # reset patience counter
        else:
            epochs_since_improvement += 1
            
            # If we exceeded the patience threshold, reduce the LR
            if epochs_since_improvement >= patience:
                for param_group in optimizer.param_groups:
                    old_lr = param_group['lr']
                    new_lr = old_lr * alpha
                    param_group['lr'] = new_lr
                epochs_since_improvement = 0  # reset counter after LR reduction
                print("New LR: ", new_lr)
        
        # Update progress bar description
        pbar.set_description(
            f"Epoch [{epoch+1}/{no_epochs}] | "
            f"Loss: {loss.item():.6f} "
            f"(PDE={loss_pde.item():.6f}, AM={loss_am.item():.6f}, Term={loss_term.item():.6f})"
        )
    
    # Load the best model weights
    best_model = copy.deepcopy(model)
    best_model.load_state_dict(best_model_weights)
    
    return best_model, losses

In [ ]:
model = DGMNet(no_assets, no_hidden, no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
best_model, losses = train_model(model, optimizer, no_epochs, int_batch_size, bd_batch_size)

### Plot Losses

In [ ]:
def plot_losses(losses, filename):
    epochs = np.arange(len(losses['res'])) + 1
    plt.figure(figsize=(15, 10))
    plt.plot(epochs, losses['res'],  color='black', linestyle=':',  label='Residual (PDE)')
    plt.plot(epochs, losses['am'],   color='black', linestyle='--', label='American')
    plt.plot(epochs, losses['term'], color='black', linestyle='-.', label='Terminal')
    #plt.yscale('log')
    plt.ylim([0, 1000])
    plt.xlabel('Epoch * Loops')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.show()

In [ ]:
filename = 'dgm_am_loss.png'
plot_losses(losses, filename)

### Test Model

In [ ]:
import QuantLib as ql

def price_american_max_basket_option(
    spots,             # list of spot prices, one for each asset
    vols,              # list of volatilities (sigma_i) for each asset
    risk_free_rate,    # constant risk-free rate
    maturity,          # in years
    strike,            # strike price K
    correlations=None, # optional correlation matrix, default = identity
    steps=50,          # time steps for MC
    seed=42            # RNG seed
):
    today = ql.Date(1, ql.January, 2024)
    ql.Settings.instance().evaluationDate = today
    day_count = ql.Actual365Fixed()
    calendar = ql.NullCalendar()
    
    maturity_date = today + int(365 * maturity)
    riskFreeCurve = ql.FlatForward(today, risk_free_rate, day_count)
    riskFreeCurveHandle = ql.YieldTermStructureHandle(riskFreeCurve)
    processes = []
    n = len(spots)
    for i in range(n):
        spot_handle = ql.QuoteHandle(ql.SimpleQuote(spots[i]))
        vol_handle  = ql.BlackVolTermStructureHandle(
            ql.BlackConstantVol(today, calendar, vols[i], day_count)
        )
        bsm_process_i = ql.BlackScholesProcess(spot_handle, riskFreeCurveHandle, vol_handle)
        processes.append(bsm_process_i)
    if correlations is None:
        # identity correlation
        correlations = [[0.0]*n for _ in range(n)]
        for i in range(n):
            correlations[i][i] = 1.0
    
    corr_matrix = ql.Matrix(n, n)
    for i in range(n):
        for j in range(n):
            corr_matrix[i][j] = correlations[i][j]
    multi_process = ql.StochasticProcessArray(processes, corr_matrix)
    
    payoff = ql.PlainVanillaPayoff(ql.Option.Call, strike)
    basket_payoff = ql.MaxBasketPayoff(payoff)  # "maximum-of" basket payoff
    exercise = ql.AmericanExercise(today, maturity_date)
    basket_option = ql.BasketOption(basket_payoff, exercise)
    basket_option.setPricingEngine(
        ql.MCAmericanBasketEngine(
            multi_process,
            "pseudorandom",
            timeSteps=steps,
            requiredTolerance=0.02,
            seed=seed,
            polynomOrder=5,
            polynomType=ql.LsmBasisSystem.Hermite,
        )
    )
    npv = basket_option.NPV()
    return npv

In [ ]:
spots = [100.0, 100.0, 100.0]

In [ ]:
price = price_american_max_basket_option(
    spots=spots,
    vols=sigmas,
    risk_free_rate=r,
    maturity=T,
    strike=K,
    correlations=None,
    steps=50,
    seed=42
)

In [ ]:
price

In [ ]:
t0 = torch.zeros((1, 1), device=device)
s0 = torch.tensor(spots, device=device).unsqueeze(dim=0)

In [ ]:
dgmprice = best_model(t0, s0)

In [ ]:
dgmprice.item()